<h1 align="center">Join Submission, Numeric and Dimension Data Sets to Find Dimensional Facts</h1>

### Import Code Libraries

In [ ]:
import polars as pl
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))

### Read Submissions, Numeric and Dimensions Data Sets for Feb. 2024 from [Financial Statement and Notes Data Sets](https://www.sec.gov/dera/data/financial-statement-and-notes-data-set)

In [ ]:
submissions = pl.read_csv('data/2024q1_notes/sub.tsv', separator='\t', infer_schema_length=10000)
numericFacts = pl.read_csv('data/2024q1_notes/num.tsv', separator='\t', infer_schema_length=10000)
dimensions = pl.read_csv('data/2024q1_notes/dim.tsv', separator='\t', infer_schema_length=10000)

### Look at the Contents of submissions DataFrame

In [ ]:
submissions

### Look at the Contents of numericFacts DataFrame

In [ ]:
numericFacts

### Look at the Contents of dimensions DataFrame

In [ ]:
dimensions

### Join submissions DataFrame with numericFacts DataFrame Using adsh Column as a Join Key

In [ ]:
mergedSubmissionsAndNumericFactsDF = submissions.join(numericFacts, on='adsh')

### Join mergedSubmissionsAndNumericFactsDF with dimensions Using Dimension Hash Column as a Join Key

In [ ]:
mergedDF = mergedSubmissionsAndNumericFactsDF.join(dimensions, left_on='dimh', right_on='dimhash')

### Look at the Contents of mergedDF DataFrame

In [ ]:
mergedDF

### Filter for Annual Revenues for Delta Airlines using RevenueFromContractWithCustomerExcludingAssessedTax tag

In [ ]:
revenueFactsForDeltaAirlines = (
    mergedDF
    .filter(
        (pl.col('name') == 'DELTA AIR LINES, INC.')
        & (pl.col('tag') == 'RevenueFromContractWithCustomerExcludingAssessedTax')
        & (pl.col('ddate') == 20231231)
        & (pl.col('qtrs') == 4)
        & (pl.col('iprx') == 0)
    )
    .select(['name', 'tag', 'dimn', 'segments', 'ddate', 'qtrs', 'uom', 'value'])
    .sort(['dimn', 'segments'])
)
revenueFactsForDeltaAirlines

### Notice that Dimensions are in the segments Column